In [37]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 1
max_iters = 10000
learning_rate = 3e-4
eval_iters = 250

cpu


In [38]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocabulary_size = len(chars)

In [39]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([92, 49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72,
        60, 63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73,
        78, 66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,
         1, 67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63,
        31, 73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77,
        63,  1, 73, 64,  1, 59, 72, 83, 73, 72])


In [40]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x.shape)
print(x)
print('targets')
print(y)

inputs:
torch.Size([1, 8])
tensor([[ 0, 37, 63, 59, 76, 67, 72, 65]])
targets
tensor([[37, 63, 59, 76, 67, 72, 65,  1]])


In [41]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, "target is", target)


when input is tensor([92]) target is tensor(49)
when input is tensor([92, 49]) target is tensor(66)
when input is tensor([92, 49, 66]) target is tensor(63)
when input is tensor([92, 49, 66, 63]) target is tensor(1)
when input is tensor([92, 49, 66, 63,  1]) target is tensor(45)
when input is tensor([92, 49, 66, 63,  1, 45]) target is tensor(76)
when input is tensor([92, 49, 66, 63,  1, 45, 76]) target is tensor(73)
when input is tensor([92, 49, 66, 63,  1, 45, 76, 73]) target is tensor(68)


In [42]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = loss.mean()
        model.train()
        return out

In [43]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, vocabulary_size)

    def forward(self, index, targets=None):

        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, loss = self.forward(index)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            index_next = torch.multinomial(probs, num_samples=1)

            index = torch.cat((index, index_next), dim=1)

        return index


model = BigramLanguageModel(vocabulary_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated_chars = decode(
    m.generate(context, max_new_tokens=500)[0].tolist()
)


    
print(context)
print(target)
print(int_to_string[context.item()])
print(int_to_string[target.item()])


first_letter = generated_chars[1]
print(first_letter)

token = string_to_int[first_letter]
print(token)


print(f" Generated Chars.start {generated_chars}")





tensor([[0]])
tensor(68)


j
.
15
 Generated Chars.start 
.o.?ZiQwJ_—xT1!”uuj3on&+*•4k5/8L#eVn﻿BGT?C"u0NL5mWWF?)
i9kWymFEdF'j‘M;kmJ[" Y8
5VJ'•VGvGb’aez”AK]PCb#x&rR]rx(
•mgXe#;!4“‘06W&BM‘3Lc
]•0i]K﻿:”,
9HOl0‘k-﻿1uOl﻿1D*Iy‘H‘9”zhiuOkUCm?_14YWVd_3‘px+&a'zrOn(R8”,1—0M[x﻿1?Za”j#*+o Z$hE#f’Li”Wo5s[.5'™Piu—S0hSJ(g'Z 4“4:eE
#*K8o“siB;vbu8YvyCr8LP4
i;n‘im•YyRLnA&2z"By$y‘RgGX﻿15v)[I
.’2GfW5Wm7O—smO,5%'+3S-s)mq?a-Rb!C9!moy‘D(9)EjJ/S/u;/N1N(Mps63LHja$-c'p”sg1• Mb‘Ht[ps ’&•[
“5Ie&S﻿l9w6Zm'™™2W9$t—N1K;7#-l;!hg]RRe?7U•0LE—IY-Ncz‘.7QnA']lON*K[6u)﻿)h6H—g$2gfh‘Rc_:to0zu


In [ ]:


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    losses = estimate_loss()
    if iter % eval_iters == 0:
        print(f"step: {iter}, losses: {losses}")
    
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step: 0, losses: {'train': tensor(4.8202)}
step: 250, losses: {'train': tensor(4.5526)}
step: 500, losses: {'train': tensor(4.7742)}
step: 750, losses: {'train': tensor(4.7596)}
step: 1000, losses: {'train': tensor(4.9414)}
step: 1250, losses: {'train': tensor(5.1200)}
step: 1500, losses: {'train': tensor(4.8660)}
step: 1750, losses: {'train': tensor(4.7994)}
step: 2000, losses: {'train': tensor(4.9476)}
step: 2250, losses: {'train': tensor(4.8789)}
step: 2500, losses: {'train': tensor(4.2757)}
step: 2750, losses: {'train': tensor(4.4778)}
step: 3000, losses: {'train': tensor(4.6861)}
step: 3250, losses: {'train': tensor(4.5983)}
step: 3500, losses: {'train': tensor(4.4289)}
step: 3750, losses: {'train': tensor(4.9227)}
step: 4000, losses: {'train': tensor(3.9575)}
step: 4250, losses: {'train': tensor(4.4557)}
step: 4500, losses: {'train': tensor(3.9273)}
step: 4750, losses: {'train': tensor(4.5143)}
step: 5000, losses: {'train': tensor(3.8307)}
step: 5250, losses: {'train': tensor(4.0

In [24]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


x5”M(Hq[m﻿j)TBa4Q%•M'A,5l•c;ffxF—VSy;cWVLWt;&es4q:B#80S—TFTYi“*0X9C1q QQgE1/X‘LMNVN4#CK”9D]6‘V[FM'﻿c2HRcT/FblKj)T/i
vgogSXs0:ctnevi﻿/x#:xe*/&QOH%,qtUM2)8,Tlgu_2[$humX,VVe_-v?*OI“9mriCk9Dzp%pZBd3Jg:4—VeseC?" Nq.-oT1TVz&cWI﻿/d,+q/62XMZ RR,X”2*L2•rerd8KPQ*:(•XPtkwb;BNC﻿#:vkK]EJoA6W.za’(/L2,kuPlOIs9s:blpQb;$“2Wj[“M'v-3ua01u4™f.D:H8$Lsq/9'7
6JRcr1/Cp2•bE,CgN•1™5i3u:"K;zQOuMZQ[fDrJg•id‘BQ-WHzZ—Qrt V+1G9j;pE# ”Gp7c;6P.”.-”DN]7USY;MQr2‘Y’]QS!g”EEYA‘dd6r™H_p5A6™xb’TSfZOW™D﻿z5™1p•ON•F:u(c!_7-mq5q:oeCSTOm,
